In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 8.0 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://marina-distance-drapery.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://marina-distance-drapery.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload) #會延續上個問題
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Ming Hsin University of Science and Technology, MHUST）坐落於新竹縣新豐鄉，是一所歷史悠久且以實務應用為導向的科技大學。

**主要特色與簡介：**

1.  **歷史沿革：** 學校前身為民國55年（1966年）創立的「明新工業專科學校」，後升格為技術學院，最終於2002年改制為科技大學。其創校精神強調「務實致用」。

2.  **地理位置優勢：** 學校鄰近新竹科學園區與多個工業區，這得天獨厚的地理位置，讓明新科大在發展產學合作、推動學生實習與畢業生就業方面，具有極大的優勢。

3.  **教育理念與特色：**
    *   **務實致用：** 學校秉持「務實致用」的教育理念，課程設計緊密結合產業需求，注重學生實作能力和專業技能的培養。
    *   **產學合作：** 與企業界建立深厚的產學合作關係，提供學生多元的實習機會、產學合作專班及專題研究，讓學生在學期間即能接觸產業實務，提升就業競爭力。
    *   **就業導向：** 學校特別強調畢業生的職場競爭力，致力於培養符合產業需求的人才，因此畢業生在就業市場上表現良好，深受企業肯定。

4.  **學院與學制：** 學校設有工程學院、管理學院、服務事業學院、設計學院等，提供學士、碩士等多層次的學位課程，涵蓋了科技、管理、服務與設計等多個專業領域。

5.  **願景：** 明新科技大學致力於成為一所理論與實務並重、學術與產業緊密結合的優質科技大學，為國家社會及產業發展培育關鍵人才。

總之，明新科技大學是一所以培育具備專業技能與人文素養、符合產業需求的科技人才為目標的學府，尤其在與產業鏈結和學生就業方面表現突出。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學現任校長是 **劉國偉** 教授。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Ua6692ca970e3a5c8679351bf1eb1a473","events":[{"type":"message","message":{"type":"text","id":"615032351307333957","quoteToken":"8EpLlG69Kloqxas4QWWc2KQKsGd9k4EWjEcF_f2EmCaStwip_SqUStPkwUr7DE4QTSlKKnQgev7XqLfyqzmO5L9cAYQp54nWWpvteiqVC1TtMFZe386tiSYjbwegiUe-mKbhs9NXGG6PveX5m9mHaQ","markAsReadToken":"q9P885S5aUxb3a7aY9mlW6uAYIqI-zuUl6Yqjvg65V9Mj6LH0Wwozl_2EbnBB-XkIRpJeDhIQc8sSBN5vtY-iJG0ViZbUCAbhVXTXfH8aal_8-2Pd5up2wMrPaD1rEKoF2Je3pliJZ8EJzI8DMdQyC1Uv8ZXkr6wFoTyewFeIM9mkRWo4MPwLvMCGjkQ_DFPOS9Ih6AHMHTHHYJeHJNVDQ","text":"AI 介紹明新30字以內"},"webhookEventId":"01KS6TAZZJYQFZZY6QYXFFC6KS","deliveryContext":{"isRedelivery":false},"timestamp":1779419152328,"source":{"type":"user","userId":"Ueea9bd8ab131bc5c9ade769c026be174"},"replyToken":"52097e21d35f4a9a8c6dc4641ec22665","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:05:54] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Ua6692ca970e3a5c8679351bf1eb1a473","events":[{"type":"message","message":{"type":"text","id":"615032396203163841","quoteToken":"meqDVbnsi_7vDlyPMCotfxGCVb91jH_tzPrtS4G2WsdvplE37zE1DRyMX0fDWWHa4_YNq31PcLAG3QAlwd9ijfF8Rq-2gWC_RE2i8s3XDS1IKY2tpKR28vzOBiRpyb-mw-c838Ncgz0g-1nyFnkLFA","markAsReadToken":"mEp7R5KSMtvuhTelBYQkV3QYxaHSFM8WJD3V9xFXCoHUpq1060KCIx1hi8XAGnTbyDASx8b1c-w2rQp3prs_Iz_4USbRgfTEMhQgKGA-4rlGpPI8k5ZX6igR5QAsDdYPK41Si9o7rwMt4UNKTe8loLRS4Pt91VQm17DS0XMwe5c1XZcYiE0j1YXjkItafbPPA6l2hF5S1T_vFmgsxUDOnQ","text":"校長是誰"},"webhookEventId":"01KS6TBTJXB5ZQMRDMF5PN1C2S","deliveryContext":{"isRedelivery":false},"timestamp":1779419179198,"source":{"type":"user","userId":"Ueea9bd8ab131bc5c9ade769c026be174"},"replyToken":"8972604b6f2f4bc49f1a3baa6d6948f0","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:06:20] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Ua6692ca970e3a5c8679351bf1eb1a473","events":[{"type":"message","message":{"type":"text","id":"615032456751349920","quoteToken":"d1J3tKbAaOsece9j3Fl5IJuVwewMgHllDB4k04w4ruHOCPKc0GsI2Hv9-1OGVyCCHhSd5rE7hrWXx6NBvGpFogGdzFy5D2btrfWjZnFvb09b72RHmiUN50fgDISrwTpWosvXKe9xAfVmOtFAoMAnrA","markAsReadToken":"fYmxs-m0Hlt6oMUP1vfjSKzNRj5Z7bNFVkfgMqgFHSv6LHyLn5-YXKTHFj7BkTKBvE_QqZffU2MukzKT0HwNCA95UqMpb8lenBO46-uA_rsFh10TL5v9-afg92MeYcwdL4evKenzB_Lna2aZE6XvEQ3IAcDHSjlWuVgHiT0GvtoGCBrBI_F-d108RERBlYe5z5MOoyULGjmfgvloYO6F4g","text":"AI 校長是誰"},"webhookEventId":"01KS6TCXVGJ5SNR5TQ25WYMS0W","deliveryContext":{"isRedelivery":false},"timestamp":1779419215261,"source":{"type":"user","userId":"Ueea9bd8ab131bc5c9ade769c026be174"},"replyToken":"94afb94eaa014b9f957089736e6dd97f","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:06:57] "POST / HTTP/1.1" 200 -
